# LAB 7 – Comparación de SRI con Precisión R

Se compara la Precisión R de dos sistemas sobre las consultas de CACM, con métricas calculadas a Z = 100 documentos:

- **A**: nuestro sistema
- **B**: sistema del equipo vecino

El archivo `CACM_PR_A_B.txt` contiene, para cada consulta, la Precisión R de A, la de B y su diferencia.
La última fila, P(PR), tiene los promedios. Se usan las 52 consultas que tienen juicios de relevancia.

In [1]:
import numpy as np

LECTURA DE DATOS

Cada línea del archivo tiene: consulta, PR_A(i), PR_B(i) y PR_A(i) − PR_B(i).

In [2]:
# devuelve [(consulta, PR_A, PR_B, diferencia)]
def leer_pr(ruta):
    filas = []
    with open(ruta, encoding="utf-8") as f:
        next(f)
        for linea in f:
            q, a, b, d = linea.split()
            if q.isdigit():
                filas.append((int(q), float(a), float(b), float(d)))
    return filas

filas = leer_pr("CACM_PR_A_B.txt")
print("Consultas comparadas:", len(filas))

Consultas comparadas: 52


TABLA DE COMPARACIÓN

Si la diferencia es positiva, A fue mejor en esa consulta; si es negativa, B fue mejor.
P(PR) es el promedio de cada columna.

In [3]:
def promedio(filas):
    return np.mean([f[1:] for f in filas], axis=0)

prom = promedio(filas)
print("%-7s %-9s %-9s %-14s" % ("Q", "PR_A(i)", "PR_B(i)", "PR_A(i)-PR_B(i)"))
for q, a, b, d in filas:
    print("%-7d %-9.4f %-9.4f %+.4f" % (q, a, b, d))
print("%-7s %-9.4f %-9.4f %+.4f" % ("P(PR)", *prom))

Q       PR_A(i)   PR_B(i)   PR_A(i)-PR_B(i)
1       0.2000    0.2000    +0.0000
2       0.0000    0.0000    +0.0000
3       0.1667    0.1667    +0.0000
4       0.1667    0.1667    +0.0000
5       0.1250    0.0000    +0.1250
6       0.0000    0.0000    +0.0000
7       0.3214    0.3214    +0.0000
8       0.0000    0.0000    +0.0000
9       0.1111    0.2222    -0.1111
10      0.6286    0.5429    +0.0857
11      0.4211    0.4211    +0.0000
12      0.4000    0.4000    +0.0000
13      0.3636    0.2727    +0.0909
14      0.4773    0.5000    -0.0227
15      0.2000    0.2000    +0.0000
16      0.1176    0.1176    +0.0000
17      0.1875    0.1250    +0.0625
18      0.1818    0.1818    +0.0000
19      0.4545    0.3636    +0.0909
20      0.0000    0.0000    +0.0000
21      0.0909    0.0000    +0.0909
22      0.4706    0.4706    +0.0000
23      0.0000    0.0000    +0.0000
24      0.1538    0.1538    +0.0000
25      0.2745    0.3137    -0.0392
26      0.4333    0.4667    -0.0334
27      0.3793    0.

ANÁLISIS DE LAS DIFERENCIAS

Cuántas consultas gana cada sistema y cuáles tienen la mayor diferencia.

In [4]:
dif = np.array([f[3] for f in filas])
print("A mejor que B:", int(np.sum(dif > 0)), "consultas")
print("B mejor que A:", int(np.sum(dif < 0)), "consultas")
print("Empate:       ", int(np.sum(dif == 0)), "consultas")
print()
print("Consultas con mayor diferencia:")
for q, a, b, d in sorted(filas, key=lambda f: abs(f[3]), reverse=True)[:5]:
    print("  Q%-3d PR_A = %.4f  PR_B = %.4f  diferencia = %+.4f" % (q, a, b, d))

A mejor que B: 10 consultas
B mejor que A: 11 consultas
Empate:        31 consultas

Consultas con mayor diferencia:
  Q64  PR_A = 0.0000  PR_B = 1.0000  diferencia = -1.0000
  Q31  PR_A = 0.5000  PR_B = 1.0000  diferencia = -0.5000
  Q5   PR_A = 0.1250  PR_B = 0.0000  diferencia = +0.1250
  Q49  PR_A = 0.1250  PR_B = 0.0000  diferencia = +0.1250
  Q9   PR_A = 0.1111  PR_B = 0.2222  diferencia = -0.1111


Cuando una consulta tiene 1 o 2 documentos relevantes, su Precisión R solo puede valer 0, 0.5 o 1,
así que una sola consulta puede mover mucho el promedio. Se calcula P(PR) sin las consultas 64 y 31.

In [5]:
for excluir in [[64], [64, 31]]:
    p = promedio([f for f in filas if f[0] not in excluir])
    print("Sin", excluir, "-> P(PR)_A = %.4f  P(PR)_B = %.4f  diferencia = %+.4f" % tuple(p))

Sin [64] -> P(PR)_A = 0.2669  P(PR)_B = 0.2693  diferencia = -0.0024
Sin [64, 31] -> P(PR)_A = 0.2623  P(PR)_B = 0.2547  diferencia = +0.0076


## Respuestas

**¿Cómo se usa la Precisión R para comparar SRI?**

La Precisión R de una consulta es la precisión en la posición |R|, donde |R| es el número de documentos relevantes de esa consulta:
se revisan los primeros |R| documentos del ranking y se divide el número de relevantes encontrados entre |R|.
Vale 1 si el sistema pone todos los relevantes en los primeros |R| lugares. Como el corte depende de cada consulta, las consultas con
muchos o pocos relevantes se miden en la misma escala.

Para comparar dos sistemas A y B se calcula PR_A(i) y PR_B(i) en cada consulta i y la diferencia PR_A(i) − PR_B(i):
si es positiva A fue mejor en esa consulta, si es negativa B fue mejor. El promedio P(PR) de cada sistema resume el desempeño global,
y el número de consultas que gana cada uno muestra si la ventaja es general o depende de unas pocas consultas.

**¿Qué sistema fue mejor?**

P(PR)_A = 0.2618 y P(PR)_B = 0.2834, con diferencia −0.0216.
**El sistema B (equipo vecino) fue ligeramente mejor.** A ganó en 10 consultas, B en 11 y empataron en 31.

**¿Por qué nuestro sistema fue peor?**

- Los dos sistemas dan resultados muy parecidos: empatan en 31 de 52 consultas y en las demás las diferencias son pequeñas y se reparten casi por igual.
- La ventaja de B viene casi toda de dos consultas con muy pocos relevantes:
  - **Q64** (1 relevante): nuestro sistema no recuperó el documento 2651, que no aparece en ninguno de los 610 documentos de nuestro ranking para esa consulta, así que PR = 0. B lo puso en primer lugar, PR = 1. Diferencia −1.0.
  - **Q31** (2 relevantes): nuestro sistema puso uno de los dos relevantes en las primeras 2 posiciones (PR = 0.5); B puso ambos (PR = 1).
- Sin Q64 la diferencia es casi cero (−0.0024), y sin Q64 ni Q31 nuestro sistema queda arriba (+0.0076).
- Lo más probable es que la diferencia venga del preprocesamiento (tokenización, stopwords, stemming): con el nuestro, el documento 2651 no comparte términos con la consulta 64.
- Nuestro sistema fue mejor en consultas como Q5, Q10, Q21, Q48 y Q49, donde B tuvo una Precisión R más baja.